In [2]:
import pandas as pd

# ======================================================
# 0. 경로 설정
# ======================================================
BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/빅데이터 알고리즘 프로젝트/IMDB/'   # 파일 위치에 맞게 수정

# ======================================================
# 1. MovieLens 로드
# ======================================================
links = pd.read_csv(BASE_PATH + 'links.csv')
movies = pd.read_csv(BASE_PATH + 'movies.csv')
ratings = pd.read_csv(BASE_PATH + 'ratings.csv')
tags = pd.read_csv(BASE_PATH + 'tags.csv')

print("MovieLens 데이터 로드 완료")

# ======================================================
# 2. imdbId → tconst 변환
# ======================================================
links['imdbId_str'] = links['imdbId'].astype(str).str.zfill(7)
links['tconst'] = 'tt' + links['imdbId_str']

print("tconst 변환 완료")


# ======================================================
# 3. IMDB basics + ratings (이 두 개는 가벼워서 그냥 읽어도 됨)
# ======================================================
basics = pd.read_csv(BASE_PATH + 'title.basics.tsv', sep='\t', na_values='\\N', low_memory=False)
ratings_imdb = pd.read_csv(BASE_PATH + 'title.ratings.tsv', sep='\t', na_values='\\N', low_memory=False)

print("basics, ratings 로드 완료")


# ======================================================
# 4. MovieLens + IMDB basics + ratings 매칭
# ======================================================
ml_imdb = links.merge(basics, on='tconst', how='left')
ml_imdb = ml_imdb.merge(ratings_imdb, on='tconst', how='left')

print("IMDB basics + ratings 매칭 완료:", ml_imdb.shape)


# ======================================================
# 5. 메모리 절약용 principals 읽기 (CHUNKS 방식)
# ======================================================
needed_tconst = set(ml_imdb['tconst'].dropna().unique())

print("필요한 tconst 개수:", len(needed_tconst))

principals_path = BASE_PATH + 'title.principals.tsv'

principals_chunks = []
chunksize = 500_000   # Colab 메모리 안전한 최대 크기

print("principals 파일을 chunk 단위로 읽는 중...")

for chunk in pd.read_csv(
    principals_path,
    sep='\t',
    na_values='\\N',
    chunksize=chunksize,
    low_memory=False
):
    # MovieLens에 등장하는 tconst만 필터링하여 메모리 절약
    filtered = chunk[chunk['tconst'].isin(needed_tconst)]
    principals_chunks.append(filtered)

principals = pd.concat(principals_chunks, ignore_index=True)

print("필요한 principals만 추출 완료:", principals.shape)


# ======================================================
# 6. directors / actors 분리
# ======================================================
directors = principals[principals['category'] == 'director']
actors = principals[principals['category'].isin(['actor', 'actress'])]

print("감독 개수:", directors.shape)
print("배우 개수:", actors.shape)


# ======================================================
# 7. MovieLens movies.csv 결합
# ======================================================
ml_full = ml_imdb.merge(movies, on='movieId', how='left')

print("MovieLens movies 결합 완료:", ml_full.shape)


# ======================================================
# 8. 저장
# ======================================================
ml_full.to_csv(BASE_PATH + 'movielens_imdb_basic.csv', index=False)
directors.to_csv(BASE_PATH + 'imdb_directors_filtered.csv', index=False)
actors.to_csv(BASE_PATH + 'imdb_actors_filtered.csv', index=False)

print("CSV 저장 완료!")

MovieLens 데이터 로드 완료
tconst 변환 완료
basics, ratings 로드 완료
IMDB basics + ratings 매칭 완료: (9742, 15)
필요한 tconst 개수: 9742
principals 파일을 chunk 단위로 읽는 중...
필요한 principals만 추출 완료: (199628, 6)
감독 개수: (10386, 6)
배우 개수: (94697, 6)
MovieLens movies 결합 완료: (9742, 17)
CSV 저장 완료!


In [3]:
# ============================================
# 1. 전체 MovieLens 영화 수
# ============================================
total_movies = links.shape[0]

# ============================================
# 2. IMDB basics가 정상적으로 매칭된 영화 수
#    (primaryTitle가 존재하면 매칭 성공으로 판단)
# ============================================
matched_movies = ml_imdb['primaryTitle'].notnull().sum()

# ============================================
# 3. 매칭 실패한 영화 수
# ============================================
unmatched_movies = total_movies - matched_movies

# ============================================
# 4. 매칭률 계산
# ============================================
match_rate = matched_movies / total_movies * 100

# ============================================
# 5. 결과 출력
# ============================================
print("✅ 전체 MovieLens 영화 수:", total_movies)
print("✅ IMDB와 매칭된 영화 수:", matched_movies)
print("❌ 매칭 실패한 영화 수:", unmatched_movies)
print(f"🎯 IMDB 매칭 성공률: {match_rate:.2f}%")

✅ 전체 MovieLens 영화 수: 9742
✅ IMDB와 매칭된 영화 수: 9719
❌ 매칭 실패한 영화 수: 23
🎯 IMDB 매칭 성공률: 99.76%


In [6]:
# IMDB와 매칭되지 않은 영화 목록
unmatched_list = ml_imdb[ml_imdb['primaryTitle'].isnull()]

print("매칭 실패 영화 예시 (전부):")
print(unmatched_list[['movieId', 'tconst']].head(25))

매칭 실패 영화 예시 (전부):
      movieId     tconst
585       720  tt0118114
2141     2851  tt0081454
3027     4051  tt0056600
3154     4241  tt0266860
3768     5264  tt0250305
4014     5672  tt0313487
4169     6003  tt0290538
4561     6776  tt0282674
5854    32600  tt0377059
6977    66511  tt1213019
7392    79677  tt1522863
7600    86668  tt1347439
7941    95771  tt0285036
8335   107780  tt2227830
8743   127194  tt3534602
9228   152173  tt0088263
9546   172875  tt0366177
9547   172881  tt0350934
9548   172887  tt0368574
9575   174551  tt0862766
9667   182639  tt0368575
9675   183197  tt7807952
9677   183227  tt7808620


In [7]:
# ============================================
# 1. IMDB와 매칭되지 않은 영화 목록 추출 (23편)
# ============================================
unmatched_movies = ml_imdb[ml_imdb['primaryTitle'].isnull()]

print("IMDB 매칭 실패 영화 수:", unmatched_movies.shape[0])

print("\nIMDB 매칭 실패 영화 목록:")
print(unmatched_movies[['movieId', 'tconst']].head(23))


# ============================================
# 2. 해당 영화들의 사용자 평점만 필터링
# ============================================
unmatched_ratings = ratings.merge(
    unmatched_movies[['movieId']],
    on='movieId',
    how='inner'
)

print("\nIMDB 매칭 실패 영화들의 전체 평점 데이터 크기:", unmatched_ratings.shape)


# ============================================
# 3. 평점 통계 요약 출력
# ============================================
rating_summary = unmatched_ratings.groupby('movieId')['rating'].agg(
    count='count',
    mean='mean',
    min='min',
    max='max'
)

print("\nIMDB 매칭 실패 영화들의 평점 통계 요약:")
print(rating_summary)


# ============================================
# 4. 평점 데이터 CSV로 저장 (보고서용)
# ============================================
unmatched_ratings.to_csv('unmatched_movie_ratings.csv', index=False)

print("\n✅ unmatched_movie_ratings.csv 저장 완료")

IMDB 매칭 실패 영화 수: 23

IMDB 매칭 실패 영화 목록:
      movieId     tconst
585       720  tt0118114
2141     2851  tt0081454
3027     4051  tt0056600
3154     4241  tt0266860
3768     5264  tt0250305
4014     5672  tt0313487
4169     6003  tt0290538
4561     6776  tt0282674
5854    32600  tt0377059
6977    66511  tt1213019
7392    79677  tt1522863
7600    86668  tt1347439
7941    95771  tt0285036
8335   107780  tt2227830
8743   127194  tt3534602
9228   152173  tt0088263
9546   172875  tt0366177
9547   172881  tt0350934
9548   172887  tt0368574
9575   174551  tt0862766
9667   182639  tt0368575
9675   183197  tt7807952
9677   183227  tt7808620

IMDB 매칭 실패 영화들의 전체 평점 데이터 크기: (77, 4)

IMDB 매칭 실패 영화들의 평점 통계 요약:
         count      mean  min  max
movieId                           
720         27  4.092593  0.5  5.0
2851         4  3.250000  2.0  5.0
4051         1  0.500000  0.5  0.5
4241         5  2.100000  0.5  3.0
5264         4  2.250000  2.0  3.0
5672         2  0.500000  0.5  0.5
6003        15 